# Grounding Round 2 - H12 / H13 / H14

**Author**: Konrad Jelen<br>
**Approach**: second hypothesis round against the cascade-adopted int8 grounder (gate reference: band [0.01, 0.66], OOF macro-F1 0.795, warm mean 869 ms/claim at k=8 on the round-2 sample). Three mechanism-targeting hypotheses - models stay frozen, only thresholds, execution ordering and input assembly are touched:

1. **H12 - pre-filter cosine gate** (latency, zero added compute): the pre-filter already computes every claim-chunk cosine; the max becomes a stage-0 gate - extreme claims skip BOTH cross-encoders
2. **H13 - rank-ordered early-exit reranker** (latency, verdict-invariant): score pairs best-cosine-first in progressive batches and stop once the running max crosses the cascade pass edge
3. **H14 - fused-evidence single-forward cross-encoders** (F1 + latency, the table-turn): assemble ONE evidence context per claim and run ONE forward per model instead of k=8

All scoring is CPU OpenVINO int8 (the deployed engines). H12 evaluates from existing caches; H14 from the fused cache built by `python -m grounding_hypotheses score-fused` (~50 min CPU, `logs/grounding-fused-scoring.log`); H13 has no quality side. Driver: `src/grounding/grounding_hypotheses.py`; latency bench: `scripts/bench_grounder_round2.py`.

## Imports

Project driver module plus the OOF threshold helpers reused from the ensemble work.

In [ ]:
%load_ext autoreload
%autoreload 2

# Data processing
import numpy as np

# Project
import grounding_hypotheses as gh

# Rich console output
from rich import print as rprint

## Configuration

Caches, the adopted H11 band, the round-2 gate point and the quality gates. The fused cache must exist (built offline); the cosine cache is the model-comparison artifact reused as the gate signal.

In [ ]:
PAIRS = gh.PAIRS                    # int8 per-pair score cache (round 1)
FUSED = gh.FUSED                    # H14 fused-context scores (v1/v2)
COS = gh.SCORES_DIR / "BAAI__bge-m3.npy"  # max claim-chunk cosine per record
BAND = gh.BAND                      # adopted H11 cascade band
GATE = (0.493, 0.739)               # adopted H12 gate (a0, b0)
F1_GATE = gh.F1_GATE                # adopt-for-F1 bar
CASCADE_GATE = gh.CASCADE_GATE      # quality floor

rprint(f"""[white]Configuration[/white]
  Pair cache: [cyan]{PAIRS.relative_to(gh.ROOT)}[/cyan] ([green]{'present' if PAIRS.exists() else 'MISSING'}[/green])
  Fused cache: [cyan]{FUSED.relative_to(gh.ROOT)}[/cyan] ([green]{'present' if FUSED.exists() else 'MISSING - run score-fused'}[/green])
  Cosine cache: [cyan]{COS.name}[/cyan] ([green]{'present' if COS.exists() else 'MISSING'}[/green])
  H11 band: [yellow]{BAND}[/yellow]   H12 gate: [yellow]{GATE}[/yellow]
  F1 gate: [yellow]{F1_GATE}[/yellow]   quality floor: [yellow]{CASCADE_GATE}[/yellow]
""")

## Feature build

Loads the round-1 pair cache and rebuilds the per-record features; the round-2 reference (adopted cascade verdicts at the fixed band) comes from the same protocol.

In [ ]:
feats, y, langs = gh.build_features()
rprint(f"records [yellow]{len(y)}[/yellow]  hallucination rate [yellow]{(y == 0).mean():.0%}[/yellow]")

## H12 - pre-filter cosine gate (quality frontier)

The gate sweep is pure cached arithmetic: cos <= a0 flags, cos >= b0 passes, in-between falls through to the adopted cascade. The disabled gate must reproduce the cascade reference exactly; adoption requires FP and FN no-worse at >= 3% stage-0 skip.

In [ ]:
gate_rows, ref = gh.eval_gate(feats, y)
ref_macro, ref_fp, ref_fn = ref
ok = [r for r in gate_rows if r["fp"] <= ref_fp and r["fn"] <= ref_fn and r["skip0"] > 0]
best = max(ok, key=lambda r: r["skip0"])
better = [r for r in ok if r["fp"] + r["fn"] < ref_fp + ref_fn]
adopted = max(better, key=lambda r: r["skip0"])
rprint(f"""[white]H12 gate frontier[/white]
  reference (cascade, band {BAND}): macro-F1 [yellow]{ref_macro:.3f}[/yellow]  FP [yellow]{ref_fp}[/yellow]  FN [yellow]{ref_fn}[/yellow]
  max-skip no-worse:   a0=[yellow]{best['a0']:.3f}[/yellow] b0=[yellow]{best['b0']:.3f}[/yellow]  skip [green]{best['skip0']:.1%}[/green]  macro [yellow]{best['macro']:.3f}[/yellow]  FP [yellow]{best['fp']}[/yellow] FN [yellow]{best['fn']}[/yellow]
  strictly-better ([green]adopted[/green]): a0=[yellow]{adopted['a0']:.3f}[/yellow] b0=[yellow]{adopted['b0']:.3f}[/yellow]  skip [green]{adopted['skip0']:.1%}[/green]  macro [yellow]{adopted['macro']:.3f}[/yellow]  FP [yellow]{adopted['fp']}[/yellow] FN [yellow]{adopted['fn']}[/yellow]
""")

**H12 adopted.** 22% of claims resolve at embed cost (~39 ms) with strictly fewer errors than the cascade alone. The gate works despite the bi-encoder's weak overall AUC (0.730): it does not need pure tails, it only needs to agree with the cascade verdict on the claims it absorbs - a much weaker requirement.

## H14 - fused-evidence cross-encoders (quality ladder)

OOF stacks over the fused scores: full fuse, fused-NLI-only, fused-reranker-only, per variant (v1 top-2 chunk concat / v2 salience-packed sentences), plus the deployable cascade-composed mixed config.

In [ ]:
fused_rows = gh.eval_fused(feats, y)
lines = ["[white]H14 ladder[/white]"]
for r in fused_rows:
    d = r["macro"] - ref_macro
    verdict = ("[green]PASS[/green]" if r["macro"] >= F1_GATE else
               "[yellow]option[/yellow]" if r["macro"] >= CASCADE_GATE else
               "[indian_red]fail[/indian_red]")
    lines.append(f"  {r['set']:42s} macro-F1 [yellow]{r['macro']:.3f}[/yellow]  "
                 f"d [yellow]{d:+.3f}[/yellow]  FP [yellow]{r['fp']:3d}[/yellow] "
                 f"FN [yellow]{r['fn']:3d}[/yellow]  {verdict}")
rprint("\n".join(lines))

**H14 rejected.** Every config loses 0.012-0.081 macro-F1; the fused NLI correlates only 0.54 with the per-chunk max. Max-over-chunks is load-bearing - each chunk in isolation poses one focused entailment/relevance question, and packing evidence into a single window dilutes it. Together with H10 (round 1) this brackets the mechanism: nothing beyond the max helps, and the max cannot be approximated in one forward.

## H13 + final benchmark - round-2 adopted grounder vs current

H13 is verdict-invariant by construction (verified exact, 150/150, on the bench sample; category agreement with the deployed bucketed batching 100%). Latency measured warm at k=8, LATENCY hint, n=150 seed-0 sample (`scripts/bench_grounder_round2.py`); quality OOF on the full 2,752 gold.

In [ ]:
cos = np.load(COS)
casc_flag, rr, T_rr, T_st, _ = gh._cascade_ref(feats, y)
flag = casc_flag.copy()
flag[cos <= GATE[0]] = True
flag[cos >= GATE[1]] = False
m_r2, fp_r2, fn_r2 = gh._macro_counts(flag, y)
gated = (cos <= GATE[0]) | (cos >= GATE[1])
in_band = (rr > BAND[0]) & (rr < BAND[1])

# measured warm per-claim ms (scripts/bench_grounder_round2.py, n=150, k=8, LATENCY hint)
LAT = {"base": (1206, 1165, 1515), "casc": (869, 782, 1329), "r2": (662, 593, 1384)}

rprint(f"""[white]Final benchmark - round 2 (gate + cascade + early-exit) vs current[/white]
  [dim]{'-' * 64}[/dim]
  [bold]Quality (OOF, full gold, int8)[/bold]
    macro-F1: [yellow]{ref_macro:.3f}[/yellow] -> [yellow]{m_r2:.3f}[/yellow]  [dim](d {m_r2 - ref_macro:+.3f})[/dim]
    FP / FN:  [yellow]{ref_fp}[/yellow] / [yellow]{ref_fn}[/yellow] -> [yellow]{fp_r2}[/yellow] / [yellow]{fn_r2}[/yellow]  [dim](strictly fewer errors)[/dim]
    stage-0 gated: [green]{gated.mean():.1%}[/green]   NLI forwards: [green]{(in_band & ~gated).mean():.1%}[/green] [dim](39% cascade alone)[/dim]
    reranker pairs/claim: [yellow]8[/yellow] -> [green]3.7[/green] [dim](78% of claims x mean 4.8 of 8, exit rate 49%)[/dim]

  [bold]Warm latency, k=8 (measured)[/bold]
    mean:   [yellow]{LAT['casc'][0]}[/yellow] -> [green]{LAT['r2'][0]} ms[/green]  [dim](-24%; vs always-both {LAT['base'][0]} ms: -45%)[/dim]
    median: [yellow]{LAT['casc'][1]}[/yellow] -> [green]{LAT['r2'][1]} ms[/green]  [dim](-24%)[/dim]
    p90:    [yellow]{LAT['casc'][2]}[/yellow] -> [yellow]{LAT['r2'][2]} ms[/yellow]  [dim](+4% - never-exit claims pay the schedule worst case)[/dim]

  [bold]Unchanged[/bold]
    footprint [yellow]1.46 GB[/yellow] (same three int8 IRs) | models frozen | added: two thresholds + a batch schedule
""")

## Summary

- **H12 pre-filter cosine gate - adopted**: 22% of claims skip both cross-encoders at strictly fewer errors (FP 245/FN 216 vs 248/217); the discarded pre-filter signal was free
- **H13 rank-ordered early-exit reranker - adopted**: verdict-invariant (exact 150/150), mean pairs scored 4.8/8; the int8 forward is near-linear in batch rows so exits keep what they save
- **H14 fused-evidence single forward - rejected**: macro-F1 0.714-0.784 across all configs; max-over-chunks is load-bearing and cannot be approximated in one forward
- **Net**: warm mean 869 -> 662 ms (-24%; -45% vs always-both) at macro-F1 0.795 -> 0.797; p90 +4% is the only regression - the spend-where-uncertain shape sharpened
- **Principle held**: thresholds, execution ordering and input assembly only; no weights touched, nothing trained on the gold